# Deploy a scikit-learn model on TrueFoundry with Triton, no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- Inference code: autogenerated by TFY CLI
- Image: build and pushed to registry first, never built by TrueFoundry
- Docs: https://www.truefoundry.com/docs/model-deployment/deploy-sklearn-xgboost

---

## 0. Setup

> **Kernel: `Python 3.12 (TFY Triton)`** — venv at `~/work/.venvs/tfy-triton-py312`.
> Not a preference: logging from 3.14 makes step 3 fail with
> `Python version 3.14 is not supported for Triton deployment`. The generated Dockerfile takes
> its base image from the *logged model version's* Python version, and the image installs
> `nvidia-pytriton`, which ships no wheels past 3.12 and needs numpy < 2.
> Documented at <https://www.truefoundry.com/docs/xgboost>: *"Python version < 3.8 and > 3.12
> is not supported for Triton deployment"*, and numpy must be < 2.0.0.

In [ ]:
# %pip install -U truefoundry scikit-learn joblib requests

In [15]:
import json
import os
import sys
import time
from pathlib import Path
from pprint import pprint

import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from truefoundry.ml import get_client, SklearnFramework, sklearn_infer_schema
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

# `!` inherits the notebook process PATH, which lacks the kernel venv's bin/.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

In [2]:
TFY_HOST = "https://internal.devtest.truefoundry.tech"
ML_REPO = "default"
CLUSTER = "tfy-usea1-devtest"
WORKSPACE = "seldon-tf"
DOMAIN = "tfy-usea1-ctl.devtest.truefoundry.tech"
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PORT = 8000
READY_TIMEOUT = 300

NAME = "iris-sk-triton"
MODEL_FILE = f"{NAME}.joblib"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [3]:
X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=10).fit(X, y)
joblib.dump(model, MODEL_FILE)

['iris-sk-triton.joblib']

In [4]:
# model.predict(X)

## 2. Log model 
- schema goes here
- `model_filepath` + `serialization_format` + `model_schema` are what make the version deployable

In [5]:
client = get_client()

framework = SklearnFramework(
    model_filepath=MODEL_FILE,
    serialization_format="joblib",
    model_schema=sklearn_infer_schema(
        model_input=X, model=model, infer_method_name="predict",
    ),
)

FQN = client.log_model(
    ml_repo=ML_REPO, name=NAME, model_file_or_folder=MODEL_FILE, framework=framework,
).fqn
print(FQN)

[truefoundry.ml] 2026-09-29T12:00:35+0100 INFO Logged in to 'https://internal.devtest.truefoundry.tech' as 'maciej.kozubal@truefoundry.com'
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:00:35+0100 INFO Logging model and additional files, this might take a while ...
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:00:35+0100 INFO Adding model file/folder to model version content
[truefoundry.ml.log_types.artifacts.utils] 2026-09-29T12:00:35+0100 INFO Adding file iris-sk-triton.joblib as /iris-sk-triton.joblib
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T12:00:35+0100 INFO Packaging and uploading files to remote with Total Size: 0.022449 MB
[truefoundry.ml] 2026-09-29T12:00:35+0100 WARNING Detected Jupyter Environment. Install `ipywidgets` to display live progress bars.
[truefoundry.ml] 2026-09-29T12:00:35+0100 INFO Uploading /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-triton.joblib to iris-sk-triton.joblib
[truefou

## 3. Generate the serving package

`tfy deploy-init` writes `server.py`, `infer.py`, `requirements.txt`, `deploy.py`, `README.md` locally.
Nothing is built and nothing is deployed — same command the UI Deploy button hands you.

In [6]:
!tfy deploy-init model --name {SVC} --model-version-fqn '{FQN}' --workspace-fqn '{WS_FQN}' --model-server triton
assert _exit_code == 0

[truefoundry.ml] 2026-09-29T12:00:45+0100 INFO Logged in to 'https://internal.devtest.truefoundry.tech' as 'maciej.kozubal@truefoundry.com'
Generating application code for 'model:truefoundry/default/iris-sk-triton:1'

Model Server code initialized successfully!

Code Location: /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-triton-svc

Next Steps:
- Navigate to the model server directory:
cd /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-sk-triton-svc
- Refer to the README file in the directory for further instructions.



In [7]:
!tree {SVC}

iris-sk-triton-svc
├── deploy.py
├── Dockerfile
├── infer.py
├── README.md
├── requirements.txt
└── server.py

1 directory, 6 files


## 4. Build and push the image

Unlike FastAPI, the Triton codegen **does** emit a `Dockerfile` (`python:3.12` + `nvidia-pytriton`),
so there is nothing to write — build it as-is and push to Docker Hub. `deploy.py` goes unused.

In [8]:
!docker build --platform linux/amd64 -t {IMAGE} {SVC}
assert _exit_code == 0

!docker push {IMAGE}
assert _exit_code == 0





[+] Building 0.0s (0/1)                                           docker:colima
[+] Building 0.2s (1/2)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 410B                                       0.0s
 => [internal] load metadata for public.ecr.aws/docker/library/python:3.1  0.2s
[+] Building 0.4s (1/2)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 410B                                       0.0s
 => [internal] load metadata for public.ecr.aws/docker/library/python:3.1  0.3s
[+] Building 0.5s (1/2)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 410B                                       0.0s
 => [internal] load metadata for pub

## 5. Deploy as a service

- image: prebuilt (`"image": {"type": "image", ...}`)
- `artifacts_download`: pulls the model version into `$MODEL_PATH` (the image is model-free)
- auth reuses the `tfy login` session (no PAT in the notebook)
- `create_or_update`: returns a deployment, so delete later with `app.data.application_id`

In [9]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "artifacts_download": {"artifacts": [{"type": "truefoundry-artifact",
        "artifact_version_fqn": FQN, "download_path_env_variable": "MODEL_PATH"}]},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

iris-sk-triton-svc gntoossvy7jsab53ms9we1zz


## 6. Call the model

- KServe v2: `POST /v2/models/<model_name>/infer`, `<model_name>` = the name `server.py` binds (no Swagger)
- Poll `/v2/models/<model_name>/ready`, **not** `/v2/health/ready` — server-ready lands ~10s before pytriton binds the model

In [10]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/v2/models/{SVC}/ready", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

iris-sk-triton-svc ready


In [16]:
r = requests.post(
    f"https://{HOST}/v2/models/{SVC}/infer",
    json={"inputs": [{"name": "input_0", "shape": [1, 4], "datatype": "FP32",
                      "data": [[5.1, 3.5, 1.4, 0.2]]}]},
    timeout=30,
)
print(f"{r.status_code = }\n")
pprint(json.loads(r.text))

r.status_code = 200

{'model_name': 'iris-sk-triton-svc',
 'model_version': '1',
 'outputs': [{'data': [0],
              'datatype': 'INT32',
              'name': 'output_0',
              'shape': [1]}]}


The same call as a copy-pasteable `curl`, for testing outside the notebook:

In [17]:
print(f'''curl -X POST https://{HOST}/v2/models/{SVC}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"inputs": [{{"name": "input_0", "shape": [1, 4], "datatype": "FP32", "data": [[5.1, 3.5, 1.4, 0.2]]}}]}}' ''')

curl -X POST https://iris-sk-triton-svc-seldon-tf-8000.tfy-usea1-ctl.devtest.truefoundry.tech/v2/models/iris-sk-triton-svc/infer \
  -H 'Content-Type: application/json' \
  -d '{"inputs": [{"name": "input_0", "shape": [1, 4], "datatype": "FP32", "data": [[5.1, 3.5, 1.4, 0.2]]}]}' 


## Cleanup
- live, not commented out — do not Run All after a deploy you want to keep
- Docker Hub delete needs `$DOCKERHUB_PAT`

In [ ]:
tfy.applications.delete(id=app.data.application_id)
client.get_model_version_by_fqn(FQN).delete()

!rm -rf {SVC} {MODEL_FILE}
!docker rmi -f {IMAGE}

# TAG is reused, so Docker Hub's copy has to go as well — a node that already pulled
# this tag will not re-pull it, and the next run would serve this image. PAT in
# $DOCKERHUB_PAT, never in the notebook.
jwt = requests.post("https://hub.docker.com/v2/users/login/",
    json={"username": DOCKER_USER, "password": os.environ["DOCKERHUB_PAT"]}).json()["token"]
r = requests.delete(f"https://hub.docker.com/v2/repositories/{DOCKER_USER}/{SVC}/tags/{TAG}/",
    headers={"Authorization": f"JWT {jwt}"})
print(r.status_code)  # 204 = gone